In [0]:
from pyspark.sql import functions as F

CATALOG = "dbw_retail_lakehouse_dev_eas_001"
LAKE_ROOT = "abfss://lakehouse@stretaildeveas001.dfs.core.windows.net"

BRONZE = f"{CATALOG}.bronze"

In [0]:
display(
    spark.table(f"{BRONZE}.customers")
)

In [0]:
products_path = (
    f"{LAKE_ROOT}/landing/products/2026/09/02/"
    "products_2026-09-02.csv"
)

df_products = (
    spark.read
    .option("header", True)
    .csv(products_path)
)

display(df_products)

In [0]:
df_products.printSchema()

In [0]:
df_products_bronze = (
    df_products
    .withColumn(
        "unit_price",
        F.col("unit_price").cast("decimal(10,2)")
    )
    .withColumn(
        "is_active",
        F.col("is_active").cast("boolean")
    )
    .withColumn(
        "_ingested_at",
        F.current_timestamp()
    )
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
)

In [0]:
display(df_products_bronze)

In [0]:
(
    df_products_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{BRONZE}.products")
)

In [0]:
spark.table(f"{BRONZE}.products").printSchema()

In [0]:
spark.table(f"{BRONZE}.products").count()

In [0]:
orders_path = (
    f"{LAKE_ROOT}/landing/orders/2026/09/02/"
    "orders_2026-09-02.csv"
)

df_orders = (
    spark.read
    .option("header", True)
    .csv(orders_path)
)

display(df_orders)

In [0]:
df_orders_bronze = (
    df_orders
    .withColumn(
        "quantity",
        F.col("quantity").cast("int")
    )
    .withColumn(
        "unit_price",
        F.col("unit_price").cast("decimal(10,2)")
    )
    .withColumn(
        "order_timestamp",
        F.to_timestamp("order_timestamp")
    )
    .withColumn(
        "_ingested_at",
        F.current_timestamp()
    )
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
)

In [0]:
display(df_orders_bronze)
df_orders_bronze.printSchema()

In [0]:
(
    df_orders_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{BRONZE}.orders")
)

In [0]:
spark.table(f"{BRONZE}.orders").count()

In [0]:
inventory_path = (
    f"{LAKE_ROOT}/landing/api/inventory/2026/09/07/"
    "inventory_page_*.json"
)

df_inventory_pages = (
    spark.read
    .json(inventory_path)
)

display(df_inventory_pages)

In [0]:
df_inventory_pages.printSchema()

In [0]:
df_inventory_bronze = (
    df_inventory_pages
    .withColumn(
        "product",
        F.explode("products")
    )
    .select(
        F.col("product.id").cast("long").alias("product_id"),
        F.col("product.title").alias("product_name"),
        F.col("product.stock").cast("int").alias("quantity_on_hand"),
        F.col("total").alias("_api_total"),
        F.col("skip").alias("_api_skip"),
        F.col("limit").alias("_api_limit"),
        F.current_timestamp().alias("_ingested_at"),
        F.col("_metadata.file_path").alias("_source_file")
    )
)

In [0]:
display(df_inventory_bronze)

In [0]:
df_inventory_bronze.count()

In [0]:
df_inventory_bronze.select(
    "product_id"
).distinct().count()

In [0]:
(
    df_inventory_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{BRONZE}.inventory")
)

In [0]:
%sql

SHOW TABLES
IN dbw_retail_lakehouse_dev_eas_001.bronze;

In [0]:
%sql
SELECT 'customers' AS table_name, COUNT(*) AS row_count
FROM dbw_retail_lakehouse_dev_eas_001.bronze.customers

UNION ALL

SELECT 'products', COUNT(*)
FROM dbw_retail_lakehouse_dev_eas_001.bronze.products

UNION ALL

SELECT 'orders', COUNT(*)
FROM dbw_retail_lakehouse_dev_eas_001.bronze.orders

UNION ALL

SELECT 'inventory', COUNT(*)
FROM dbw_retail_lakehouse_dev_eas_001.bronze.inventory;